# Bank Marketing - Prédiction de souscription à un crédit

## Introduction

### Contexte

Dans un secteur bancaire de plus en plus compétitif, les institutions financières cherchent à optimiser leurs campagnes marketing pour attirer des clients fiables et rentables. Parmi les produits stratégiques proposés par les banques, le dépôt à terme est un placement à faible risque qui garantit des revenus pour la banque.

Ce projet s’appuie sur un dataset issu d’une banque portugaise, contenant les résultats de campagnes de marketing téléphonique visant à promouvoir la souscription à des dépôts à terme.

L’objectif est de prédire, à partir du profil du client, si celui-ci est susceptible de souscrire à cette offre, afin de mieux cibler les actions marketing futures.

## Présentation des Données

Deux fichiers sont utilisés dans cette étude :

- `bank-full.csv` : contient **45 211** enregistrements correspondant à l’ensemble des campagnes marketing menées.
- `bank.csv` : échantillon plus restreint de **4 521** enregistrements, avec la même structure.

Chaque ligne représente un contact individuel entre la banque et un client, avec des informations sociodémographiques, financières, et comportementales, ainsi que le résultat de la campagne.

### Caractéristiques principales du dataset :

| Colonne   | Description |
|-----------|-------------|
| age       | Âge du client |
| job       | Profession (admin., technician, unemployed...) |
| marital   | État civil (married, single, divorced) |
| education | Niveau d’éducation |
| default   | A déjà fait défaut sur un crédit ? (yes/no) |
| balance   | Solde bancaire annuel |
| housing   | A un prêt immobilier ? |
| loan      | A un prêt personnel ? |
| contact   | Type de contact (cellulaire, téléphone fixe) |
| day       | Jour du dernier contact |
| month     | Mois du dernier contact |
| duration  | Durée du dernier appel en secondes |
| campaign  | Nombre de contacts durant cette campagne |
| pdays     | Jours depuis le dernier contact dans une campagne précédente (-1 = jamais contacté) |
| previous  | Nombre de contacts précédents |
| poutcome  | Résultat de la précédente campagne |
| y         | Variable cible : a-t-il souscrit ? (yes / no) |

## Approche Méthodologique

### Stratégie d'Analyse

Cette étude adopte une approche d'apprentissage automatique supervisé pour résoudre un problème de classification binaire. L'utilisation de **PyCaret**, une librairie AutoML, permettra d'automatiser le processus de développement de modèles et d'optimiser les performances prédictives.

### Étapes du Projet

- **Exploration et nettoyage des données** : Identification et traitement des anomalies  
- **Analyse exploratoire** : Compréhension des patterns et relations dans les données  
- **Prétraitement** : Préparation des données pour l'entraînement des modèles  
- **Modélisation automatisée** : Entraînement et comparaison de multiples algorithmes  
- **Évaluation et sélection** : Identification du meilleur modèle basé sur les métriques de performance  
- **Interprétation des résultats** : Analyse des facteurs prédictifs les plus importants  

## Enjeux et Applications

### Impact Business

La capacité à prédire avec précision la propension d'un client à souscrire un dépôt à terme représente un avantage concurrentiel significatif pour les institutions bancaires dans l'optimisation de leurs campagnes marketing.

### Contribution Technique

Ce projet illustre l'application pratique de l'AutoML dans un contexte business réel, démontrant comment les techniques d'apprentissage automatique peuvent être rendues accessibles et efficaces pour résoudre des problèmes de classification complexes.

La méthodologie développée pourra être réutilisée et adaptée pour d'autres cas d'usage similaires dans le domaine du marketing prédictif.

In [ ]:
### Imports

In [59]:
import pandas as pd

## Chargement des fichiers CSV 

In [66]:
# Charger les fichiers CSV
df_large = pd.read_csv('../data/raw/bank-full.csv', sep=';')
df_small = pd.read_csv('../data/raw/bank.csv', sep=';')

## Exploration des datasets

### Exploration df_large

In [99]:
df_large.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


#### Recherche de doublons

In [16]:
# Vérifier le nombre de doublons dans le DataFrame
num_duplicates = df_large.duplicated().sum()

# Afficher le nombre de doublons
print(f"Nombre de doublons dans le DataFrame : {num_duplicates}")

Nombre de doublons dans le DataFrame : 0


#### **Analyse Structurelle : Métadonnées et Intégrité du Dataset**

In [12]:
info_large = df_large.info()
print(info_large)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45211 entries, 0 to 45210
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        45211 non-null  int64 
 1   job        45211 non-null  object
 2   marital    45211 non-null  object
 3   education  45211 non-null  object
 4   default    45211 non-null  object
 5   balance    45211 non-null  int64 
 6   housing    45211 non-null  object
 7   loan       45211 non-null  object
 8   contact    45211 non-null  object
 9   day        45211 non-null  int64 
 10  month      45211 non-null  object
 11  duration   45211 non-null  int64 
 12  campaign   45211 non-null  int64 
 13  pdays      45211 non-null  int64 
 14  previous   45211 non-null  int64 
 15  poutcome   45211 non-null  object
 16  y          45211 non-null  object
dtypes: int64(7), object(10)
memory usage: 5.9+ MB
None


#### **1. Complétude des Données**
  **Aucune valeur manquante détectée**  
- Toutes les colonnes présentent un `Non-Null Count` égal au nombre total d'entrées (`45 211`)  
- **Implication** :  
  - Aucun traitement de imputation nécessaire  
  - Données immédiatement exploitables pour l'analyse  

#### **2. Cohérence des Types de Données**  
  **Typage conforme aux attentes**  
- **Variables numériques** :  
  - `age`, `balance`, `duration` correctement en `int64`/`float64`  
- **Variables catégorielles** :  
  - `job`, `education`, etc. bien typées en `object`  
- **Variables binaires** :  
  - `default`, `housing` encodées en texte (`object`)  

#### **Analyse Exploratoire Initiale : Statistiques Descriptives**

In [11]:
description_large = df_large.describe()
print(description_large)

                age        balance           day      duration      campaign  \
count  45211.000000   45211.000000  45211.000000  45211.000000  45211.000000   
mean      40.936210    1362.272058     15.806419    258.163080      2.763841   
std       10.618762    3044.765829      8.322476    257.527812      3.098021   
min       18.000000   -8019.000000      1.000000      0.000000      1.000000   
25%       33.000000      72.000000      8.000000    103.000000      1.000000   
50%       39.000000     448.000000     16.000000    180.000000      2.000000   
75%       48.000000    1428.000000     21.000000    319.000000      3.000000   
max       95.000000  102127.000000     31.000000   4918.000000     63.000000   

              pdays      previous  
count  45211.000000  45211.000000  
mean      40.197828      0.580323  
std      100.128746      2.303441  
min       -1.000000      0.000000  
25%       -1.000000      0.000000  
50%       -1.000000      0.000000  
75%       -1.000000      0.

### Analyse des principales observations

#### 1. Colonne "campaign" (nombre de contacts)
- **Moyenne** : 3 contacts
- **Écart-type** : 3 (forte dispersion)
- **Distribution** :
  - 75% des clients ont ≤3 contacts
  - Présence possible de valeurs extrêmes

#### 2. Colonne "duration" (durée d'appel)
- **Problèmes** :
  - Valeurs min=0 (non plausibles)
  - Valeurs max trop élevées
- **Recommandation** :
  - Filtrer les durées=0
  - Examiner les outliers

#### 3. Colonne "contact"
- Peu informative dans sa forme actuelle
- Possible suppression

#### 4. Colonne "previous"
- Valeur max=275 (outlier probable)
- À investiguer

#### 5. Colonnes "balance" et "age"
- Distributions cohérentes
- Pas d'anomalie détectée

### Analyse colonne par colonne

#### "job"

In [46]:
print(df_large['job'].value_counts())

job
blue-collar      9732
management       9458
technician       7597
admin.           5171
services         4154
retired          2264
self-employed    1579
entrepreneur     1487
unemployed       1303
housemaid        1240
student           938
unknown           288
Name: count, dtype: int64


Observations: 288 valeurs inconnus

#### "education"

In [47]:
print(df_large['education'].value_counts())

education
secondary    23202
tertiary     13301
primary       6851
unknown       1857
Name: count, dtype: int64


Observations: 1857 valeurs inconnus

#### 'contact'

In [48]:
print(df_large['contact'].value_counts())

contact
cellular     29285
unknown      13020
telephone     2906
Name: count, dtype: int64


In [ ]:
Observations: 13020 valeurs inconnus

#### 'day'

In [51]:
print(df_large['day'].value_counts())

day
20    2752
18    2308
21    2026
17    1939
6     1932
5     1910
14    1848
8     1842
28    1830
7     1817
19    1757
29    1745
15    1703
12    1603
13    1585
30    1566
9     1561
11    1479
4     1445
16    1415
2     1293
27    1121
3     1079
26    1035
23     939
22     905
25     840
31     643
10     524
24     447
1      322
Name: count, dtype: int64


Segmentation en début de mois et fin de mois

In [69]:
# Compter les entrées entre 1 et 15 inclus
count_1_15 = len(df_large[(df_large['day'] >= 1) & (df_large['day'] <= 15)])

# Compter les entrées entre 16 et 31 inclus
count_16_31 = len(df_large[(df_large['day'] >= 16) & (df_large['day'] <= 31)])

print(f"Nombre d'entrées pour day 1-15 : {count_1_15}")
print(f"Nombre d'entrées pour day 16-31 : {count_16_31}")

Nombre d'entrées pour day 1-15 : 21943
Nombre d'entrées pour day 16-31 : 23268


observations: Possibilité de partager en deux parties quasi-éguales 

#### 'month'

In [52]:
print(df_large['month'].value_counts())

month
may    13766
jul     6895
aug     6247
jun     5341
nov     3970
apr     2932
feb     2649
jan     1403
oct      738
sep      579
mar      477
dec      214
Name: count, dtype: int64


In [ ]:
Observations: Mauvaise répartition des entrées en fonction des mois

#### 'pdays'

In [54]:
print(df_large['pdays'].value_counts())

pdays
-1      36954
 182      167
 92       147
 91       126
 183      126
        ...  
 449        1
 452        1
 648        1
 595        1
 530        1
Name: count, Length: 559, dtype: int64


Nombre d'entrée au dessus de 365 jours

In [56]:
count_above_10 = (df_large['pdays'] > 365).sum()
print(count_above_10)

643


Observations: 
- Une grande partie des entrées sont un premier contact.
- Certaines valeurs trop hautes sont unique (643 supérieures à 1 an)

#### 'previous'

In [57]:
print(df_large['previous'].value_counts())

previous
0      36954
1       2772
2       2106
3       1142
4        714
5        459
6        277
7        205
8        129
9         92
10        67
11        65
12        44
13        38
15        20
14        19
17        15
16        13
19        11
20         8
23         8
18         6
22         6
24         5
27         5
21         4
29         4
25         4
30         3
38         2
37         2
26         2
28         2
51         1
275        1
58         1
32         1
40         1
55         1
35         1
41         1
Name: count, dtype: int64


Nombre d'entrées avec une valeurs au dessus de 30

In [93]:
count_above_30 = (df_large['previous'] > 30).sum()
print(count_above_30)

12


Observations: 
- On retrouve le même nombre d'entrées en premier contact
- Certaines valeurs hautes n'ont peu d'occurences  

#### 'campaign'

In [72]:
print(df_large['campaign'].value_counts())

campaign
1     17544
2     12505
3      5521
4      3522
5      1764
6      1291
7       735
8       540
9       327
10      266
11      201
12      155
13      133
14       93
15       84
16       79
17       69
18       51
19       44
20       43
21       35
22       23
25       22
23       22
24       20
29       16
28       16
26       13
31       12
27       10
32        9
30        8
33        6
34        5
36        4
35        4
43        3
38        3
37        2
50        2
41        2
46        1
58        1
55        1
63        1
51        1
39        1
44        1
Name: count, dtype: int64


In [45]:
count_above_10 = (df_large['campaign'] > 10).sum()
print(count_above_10)

1196


Exploration de la quantité de conversion pour les entrée avec plus de 30 appels

In [92]:
# Filtrer les entrées avec campaign > 30
high_campaign = df_large[df_large['campaign'] > 30]

#  Compter les occurrences de 'y'
counts = high_campaign['y'].value_counts()

#  Affichage
print("Répartition de 'y' pour les entrées avec campaign > 30 :")
print(f"Nombre total de clients concernés : {len(high_campaign)}")
print("\nDécompte :")
print(f"- 'no'  : {counts.get('no', 0)} occurrences")
print(f"- 'yes' : {counts.get('yes', 0)} occurrences")

Répartition de 'y' pour les entrées avec campaign > 30 :
Nombre total de clients concernés : 59

Décompte :
- 'no'  : 58 occurrences
- 'yes' : 1 occurrences


Observations :
- 1196 entrées avec un nombres d'appels supérieur à 10 pour cette campagne

#### 'poutcome'

In [73]:
print(df_large['poutcome'].value_counts())

poutcome
unknown    36959
failure     4901
other       1840
success     1511
Name: count, dtype: int64


Observations: 
- Une grande majorité d'entrées avec valeur inconnu

#### 'duration'

In [74]:
print(df_large['duration'].value_counts())

duration
124     188
90      184
89      177
104     175
122     175
       ... 
1833      1
1545      1
1352      1
1342      1
1556      1
Name: count, Length: 1573, dtype: int64


In [79]:
count_above_1000 = (df_large['duration'] >900 ).sum()
print(count_above_1000)

1418


In [77]:
count_0 = (df_large['duration'] == 0 ).sum()
print(count_0)

3


Observations: 
- Répartitions des valeurs trés hétérogènes
- 3 entrées à 0 => Incohérence
- 1418 entrées au dessus de 16 minutes de communication

#### 'loan'

In [80]:
print(df_large['loan'].value_counts())

loan
no     37967
yes     7244
Name: count, dtype: int64


Observations: Pas de présences de valeurs intendues

#### 'housing'

In [81]:
print(df_large['housing'].value_counts())

housing
yes    25130
no     20081
Name: count, dtype: int64


Observations: Pas de présences de valeurs intendues

### Exploration df_small

In [100]:
df_small.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,30,unemployed,married,primary,no,1787,no,no,cellular,19,oct,79,1,-1,0,unknown,no
1,33,services,married,secondary,no,4789,yes,yes,cellular,11,may,220,1,339,4,failure,no
2,35,management,single,tertiary,no,1350,yes,no,cellular,16,apr,185,1,330,1,failure,no
3,30,management,married,tertiary,no,1476,yes,yes,unknown,3,jun,199,4,-1,0,unknown,no
4,59,blue-collar,married,secondary,no,0,yes,no,unknown,5,may,226,1,-1,0,unknown,no


#### Recherche de doublons

In [97]:
# Vérifier le nombre de doublons dans le DataFrame
num_duplicates = df_small.duplicated().sum()

# Afficher le nombre de doublons
print(f"Nombre de doublons dans le DataFrame : {num_duplicates}")

Nombre de doublons dans le DataFrame : 0


#### **Analyse Structurelle : Métadonnées et Intégrité du Dataset**

In [102]:
info_small = df_small.info()
print(info_small)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4521 entries, 0 to 4520
Data columns (total 17 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        4521 non-null   int64 
 1   job        4521 non-null   object
 2   marital    4521 non-null   object
 3   education  4521 non-null   object
 4   default    4521 non-null   object
 5   balance    4521 non-null   int64 
 6   housing    4521 non-null   object
 7   loan       4521 non-null   object
 8   contact    4521 non-null   object
 9   day        4521 non-null   int64 
 10  month      4521 non-null   object
 11  duration   4521 non-null   int64 
 12  campaign   4521 non-null   int64 
 13  pdays      4521 non-null   int64 
 14  previous   4521 non-null   int64 
 15  poutcome   4521 non-null   object
 16  y          4521 non-null   object
dtypes: int64(7), object(10)
memory usage: 600.6+ KB
None


#### **1. Complétude des Données**
  **Aucune valeur manquante détectée**  
- Toutes les colonnes présentent un `Non-Null Count` égal au nombre total d'entrées (`45 211`)  
- **Implication** :  
  - Aucun traitement de imputation nécessaire  
  - Données immédiatement exploitables pour l'analyse  

#### **2. Cohérence des Types de Données**  
  **Typage conforme aux attentes**  
- **Variables numériques** :  
  - `age`, `balance`, `duration` correctement en `int64`/`float64`  
- **Variables catégorielles** :  
  - `job`, `education`, etc. bien typées en `object`  
- **Variables binaires** :  
  - `default`, `housing` encodées en texte (`object`)  

#### **Analyse Exploratoire Initiale : Statistiques Descriptives**

In [103]:
description_small = df_small.describe()
print(description_small)

               age       balance          day     duration     campaign  \
count  4521.000000   4521.000000  4521.000000  4521.000000  4521.000000   
mean     41.170095   1422.657819    15.915284   263.961292     2.793630   
std      10.576211   3009.638142     8.247667   259.856633     3.109807   
min      19.000000  -3313.000000     1.000000     4.000000     1.000000   
25%      33.000000     69.000000     9.000000   104.000000     1.000000   
50%      39.000000    444.000000    16.000000   185.000000     2.000000   
75%      49.000000   1480.000000    21.000000   329.000000     3.000000   
max      87.000000  71188.000000    31.000000  3025.000000    50.000000   

             pdays     previous  
count  4521.000000  4521.000000  
mean     39.766645     0.542579  
std     100.121124     1.693562  
min      -1.000000     0.000000  
25%      -1.000000     0.000000  
50%      -1.000000     0.000000  
75%      -1.000000     0.000000  
max     871.000000    25.000000  


### Analyse des principales observations

#### 1. Colonne "campaign" (nombre de contacts)
- **Moyenne** : 3 contacts
- **Écart-type** : 3 (forte dispersion)
- **Distribution** :
  - 75% des clients ont ≤3 contacts
  - Présence possible de valeurs extrêmes

#### 2. Colonne "duration" (durée d'appel)
- **Problèmes** :
  - Valeurs max trop élevées
- **Recommandation** :
  - Filtrer les durées=0
  - Examiner les outliers

#### 3. Colonnes "balance" et "age"
- Distributions cohérentes
- Pas d'anomalie détectée

### Analyse colonne par colonne

#### "job"

In [104]:
print(df_small['job'].value_counts())

job
management       969
blue-collar      946
technician       768
admin.           478
services         417
retired          230
self-employed    183
entrepreneur     168
unemployed       128
housemaid        112
student           84
unknown           38
Name: count, dtype: int64


Observations: Conforme à df_large

#### "education"

In [105]:
print(df_small['education'].value_counts())

education
secondary    2306
tertiary     1350
primary       678
unknown       187
Name: count, dtype: int64


Observations: Conforme à df_large

#### 'contact'

In [107]:
print(df_small['contact'].value_counts())

contact
cellular     2896
unknown      1324
telephone     301
Name: count, dtype: int64


Observations: Conforme à df_large

#### 'day'

In [108]:
print(df_small['day'].value_counts())

day
20    257
18    226
19    201
21    198
14    195
17    191
7     190
6     187
28    181
5     181
8     180
29    175
15    174
30    168
13    166
16    164
9     163
11    152
12    151
4     139
2     114
27    113
26    110
3     105
23    102
22     86
25     80
31     59
10     50
24     36
1      27
Name: count, dtype: int64


Observations: Conforme à df_large

#### 'month'

In [109]:
print(df_small['month'].value_counts())

month
may    1398
jul     706
aug     633
jun     531
nov     389
apr     293
feb     222
jan     148
oct      80
sep      52
mar      49
dec      20
Name: count, dtype: int64


Observations: Conforme à df_large

#### 'pdays'

In [110]:
print(df_small['pdays'].value_counts())

pdays
-1      3705
 182      23
 183      20
 363      12
 92       12
        ... 
 118       1
 386       1
 63        1
 81        1
 234       1
Name: count, Length: 292, dtype: int64


Observations: Conforme à df_large

#### 'previous'

In [111]:
print(df_small['previous'].value_counts())

previous
0     3705
1      286
2      193
3      113
4       78
5       47
6       25
7       22
8       18
9       10
12       5
10       4
11       3
14       2
24       1
22       1
23       1
17       1
18       1
15       1
13       1
19       1
20       1
25       1
Name: count, dtype: int64


Observations: Conforme à df_large

#### 'campaign'

In [111]:
print(df_small['campaign'].value_counts())

previous
0     3705
1      286
2      193
3      113
4       78
5       47
6       25
7       22
8       18
9       10
12       5
10       4
11       3
14       2
24       1
22       1
23       1
17       1
18       1
15       1
13       1
19       1
20       1
25       1
Name: count, dtype: int64


Observations: Conforme à df_large

#### 'poutcome'

In [112]:
print(df_small['poutcome'].value_counts())

poutcome
unknown    3705
failure     490
other       197
success     129
Name: count, dtype: int64


Observations: Conforme à df_large

#### 'loan'

In [113]:
print(df_small['loan'].value_counts())

loan
no     3830
yes     691
Name: count, dtype: int64


Observations: Conforme à df_large

#### 'housing'

In [114]:
print(df_small['housing'].value_counts())

housing
yes    2559
no     1962
Name: count, dtype: int64


Observations: Conforme à df_large

#### Conclusion de nos observations de ce dataset

# Préparation des données

## Pour les colonnes caractérisant le type de client :

- **La colonne age** est à garder en l'état
- **La colonne job** : supprimer les entrées "unknown"
- **La colonne education** : NE PAS supprimer les 1857 entrées "unknown". Elles représentent plus de 4% du dataset et leur suppression constituerait une perte massive d'information. Garder "unknown" comme catégorie à part entière.
- **La colonne balance** : NE PAS supprimer les clients en solde négatif annuel. Les soldes négatifs constituent un signal prédictif crucial car ils peuvent influencer la souscription. Supprimer ces données reviendrait à censurer un comportement client important.
- **La colonne marital** est à garder en l'état
- **La colonne default** : NE PAS supprimer les clients ayant eu un défaut de paiement sur un crédit. L'historique de défaut peut influencer la décision de souscription et doit être conservé comme facteur explicatif.
- **Les colonnes loan et housing** à garder en l'état

## Pour les colonnes relatives aux campagnes marketing :

- **La colonne contact** : analyser avant de supprimer car le type de contact peut influencer le taux de conversion (mobile vs fixe = différence de joignabilité). Recommandation : analyser d'abord, puis éventuellement encoder.
- **La colonne day** est à garder en l'état et segmenter par la suite pour vérifier que certains moments du mois sont plus propices à la signature du crédit
- **La colonne month** à garder pour une étude spécifique de l'efficacité de la campagne selon le mois.
- **La colonne duration** : Elle fuit le résultat (on ne peut pas connaître la durée d'un appel avant de le faire). La colonne doit être retirée du modèle final.
- **La colonne poutcome** DOIT ETRE GARDEE. Même si il y a une énorme occurrence des "unknown", elle permet de voir le comportement de certains clients dont on a les valeurs et les catégories "success" et "failure" sont très prédictives.
- **La colonne previous** : garder l'entrée avec une valeur à 275. Cet outlier peut contenir de l'information utile. Pour le reste, le faible nombre d'entrées avec des valeurs élevées peut être gardé, cela peut permettre de vérifier si le nombre d'appels compte.
- **La colonne pdays** : NE PAS garder en l'état. Les valeurs -1 doivent obligatoirement être traitées car elles signifient "jamais contacté précédemment".

## Note sur les colonnes binaires :

- **Les colonnes binaires (yes/no)** comme default, housing, loan, y sont gardées en format object. PyCaret gère automatiquement la conversion de ces variables lors de l'entraînement.
